# SAEs Reveal Algorithmic Representations in NARs — Experiments

End-to-end pipeline:
1. **Phase 1**: Train NAR on CLRS-30 (BFS first)
2. **Phase 2**: Collect activations & train SAE
3. **Phase 3**: Feature interpretation & concept correlations

In [ ]:
# --- Colab setup (skip if running locally) ---
import os

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(os.environ.get("COLAB_RELEASE_TAG", ""))

if IN_COLAB:
    !git clone https://github.com/aniervs/nar-mechinterp.git
    %cd nar-mechinterp
    !pip install -e . 2>&1 | tail -5
    !pip install salsa-clrs@git+https://github.com/jkminder/SALSA-CLRS.git 2>&1 | tail -3
    print("Colab setup complete!")
else:
    print("Running locally, skipping Colab setup.")

In [ ]:
import sys
from pathlib import Path

# Handle import paths for both Colab (cwd=repo root) and local (cwd=experiments/)
if Path("../data").exists() and ".." not in sys.path:
    sys.path.insert(0, "..")
elif Path("data").exists() and "." not in sys.path:
    sys.path.insert(0, ".")

import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import numpy as np

from data import (
    get_clrs_dataset,
    get_clrs_dataloader,
    get_algorithm_spec,
    spec_to_model_types,
    batch_to_model_inputs,
)
from models import NARModel
from interp import (
    SparseAutoencoder, BatchTopKSAE, Transcoder,
    SAETrainer, SAEOutput,
    ActivationCollector, make_activation_dataloader,
    FeatureAnalyzer, FeatureAnalysisResult,
)
from interp.concept_labels import collect_concept_labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

SEED = 42
torch.manual_seed(SEED)

## Configuration

In [ ]:
# --- Experiment configuration ---
ALGORITHM = "bfs"  # Start with BFS (simplest, clearest ground-truth concepts)

# NAR model hyperparameters
HIDDEN_DIM = 128
NUM_LAYERS = 4
NUM_HEADS = 8
PROCESSOR_TYPE = "mpnn"

# Training
NAR_EPOCHS = 100
NAR_BATCH_SIZE = 32
NAR_LR = 1e-3
TRAIN_SAMPLES = 1000
VAL_SAMPLES = 128
NUM_NODES = 16
EDGE_PROB = 0.2

# SAE hyperparameters
SAE_TYPE = "batchtopk"  # Primary variant (recommended over standard)
EXPANSION_FACTOR = 8     # dict_size = HIDDEN_DIM * EXPANSION_FACTOR = 1024
SAE_K = 32               # Average active features per sample (BatchTopK)
SAE_LR = 3e-4
SAE_BATCH_SIZE = 256
SAE_STEPS = 50_000
SAE_NUM_SAMPLES = 10_000  # CLRS samples for activation collection

# Paths — work from repo root (Colab) or experiments/ (local)
REPO_ROOT = Path("..") if Path("../data").exists() else Path(".")
CHECKPOINT_DIR = REPO_ROOT / "checkpoints" / ALGORITHM
RESULTS_DIR = REPO_ROOT / "results" / "sae" / ALGORITHM / SAE_TYPE
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Algorithm: {ALGORITHM}")
print(f"NAR: hidden_dim={HIDDEN_DIM}, layers={NUM_LAYERS}, heads={NUM_HEADS}, processor={PROCESSOR_TYPE}")
print(f"SAE: type={SAE_TYPE}, expansion={EXPANSION_FACTOR}x, dict_size={HIDDEN_DIM * EXPANSION_FACTOR}, k={SAE_K}")
print(f"Checkpoints: {CHECKPOINT_DIR.resolve()}")
print(f"Results: {RESULTS_DIR.resolve()}")

---
## Phase 1: Train NAR on CLRS-30

Train an MPNN-based NAR model on BFS. The model learns to predict BFS outputs (reachability) with hint supervision at each message-passing step.

### 1.1 Load CLRS-30 dataset

In [ ]:
# Load dataset and extract algorithm spec
ds = get_clrs_dataset(
    ALGORITHM, split="train",
    num_samples=TRAIN_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
)
spec = get_algorithm_spec(ALGORITHM, dataset=ds)
output_types, hint_types = spec_to_model_types(spec)

print(f"Output types: {output_types}")
print(f"Hint types: {hint_types}")

# Create dataloaders
train_loader = get_clrs_dataloader(
    ALGORITHM, "train",
    batch_size=NAR_BATCH_SIZE,
    num_samples=TRAIN_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED,
)
val_loader = get_clrs_dataloader(
    ALGORITHM, "val",
    batch_size=NAR_BATCH_SIZE,
    num_samples=VAL_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED + 1,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

### 1.2 Train NAR model

In [ ]:
model = NARModel(
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    processor_type=PROCESSOR_TYPE,
).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"NAR model parameters: {num_params:,}")

In [ ]:
def train_epoch(model, loader, optimizer, spec, output_types, hint_types, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        inputs, outputs, hints = batch_to_model_inputs(batch, spec, device)
        optimizer.zero_grad()
        output = model(
            inputs=inputs, hints=hints, outputs=outputs,
            output_types=output_types, hint_types=hint_types,
            num_steps=batch.lengths.max().item(),
        )
        output.total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += output.total_loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, spec, output_types, hint_types, device):
    model.eval()
    total_loss = 0
    total_acc = 0
    count = 0
    for batch in loader:
        batch = batch.to(device)
        inputs, outputs, hints = batch_to_model_inputs(batch, spec, device)
        output = model(
            inputs=inputs, outputs=outputs,
            output_types=output_types, hint_types=hint_types,
            num_steps=batch.lengths.max().item(),
        )
        total_loss += output.total_loss.item()
        for name, pred in output.predictions.items():
            if name in outputs:
                target = outputs[name]
                if output_types.get(name) == 'node_mask':
                    pred_bin = torch.sigmoid(pred[:, :target.shape[-1]]) > 0.5
                    acc = (pred_bin == target.bool()).float().mean()
                elif output_types.get(name) == 'node_pointer':
                    pred_idx = pred.argmax(-1)
                    tgt_idx = target.long() if target.dim() < pred.dim() else target.argmax(-1)
                    acc = (pred_idx == tgt_idx).float().mean()
                else:
                    acc = torch.tensor(0.0)
                total_acc += acc.item()
                count += 1
    return total_loss / max(len(loader), 1), total_acc / max(count, 1)

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=NAR_LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NAR_EPOCHS)

train_losses, val_losses, val_accs = [], [], []
best_loss = float('inf')

for epoch in tqdm(range(NAR_EPOCHS), desc="Training NAR"):
    train_loss = train_epoch(model, train_loader, optimizer, spec, output_types, hint_types, device)
    val_loss, val_acc = validate(model, val_loader, spec, output_types, hint_types, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'config': {
                'hidden_dim': HIDDEN_DIM, 'num_layers': NUM_LAYERS,
                'num_heads': NUM_HEADS, 'processor_type': PROCESSOR_TYPE,
            },
        }, CHECKPOINT_DIR / "best.pt")

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{NAR_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Acc: {val_acc:.4f}")

torch.save({'model_state_dict': model.state_dict()}, CHECKPOINT_DIR / "final.pt")
print(f"\nBest val loss: {best_loss:.4f}")

### 1.3 Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label="Train")
ax1.plot(val_losses, label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title(f"NAR Training — {ALGORITHM.upper()}")
ax1.legend()
ax1.set_yscale("log")

ax2.plot(val_accs)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Accuracy")
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## Phase 2: Collect Activations & Train SAE

Collect processor hidden states from the trained NAR, then train a BatchTopK SAE to discover monosemantic features. Each (node, message-passing step) pair is treated as an independent sample.

### 2.1 Load best NAR checkpoint

In [ ]:
# Load best checkpoint
ckpt = torch.load(CHECKPOINT_DIR / "best.pt", map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}, val_acc={ckpt['val_acc']:.4f})")

### 2.2 Collect processor activations

In [ ]:
# Create a larger dataset for activation collection
act_loader = get_clrs_dataloader(
    ALGORITHM, "train",
    batch_size=32,
    num_samples=SAE_NUM_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED + 100,
)

collector = ActivationCollector(model, spec=spec, device=device)
activations = collector.collect(act_loader, output_types=output_types)
print(f"Collected activations: {activations.shape}")
print(f"  = {activations.shape[0]:,} activation vectors of dim {activations.shape[1]}")

# Save activations
torch.save(activations, RESULTS_DIR / "activations.pt")
print(f"Saved to {RESULTS_DIR / 'activations.pt'}")

### 2.3 Collect concept labels

Extract ground-truth algorithmic concept labels (is_visited, is_frontier, is_source, etc.) aligned with the activation vectors.

In [ ]:
# Recreate dataloader with same seed (no shuffle) so labels align with activations
label_loader = get_clrs_dataloader(
    ALGORITHM, "train",
    batch_size=32,
    num_samples=SAE_NUM_SAMPLES,
    num_nodes=NUM_NODES,
    edge_probability=EDGE_PROB,
    seed=SEED + 100,
    shuffle=False,
)

concept_result = collect_concept_labels(label_loader, spec, ALGORITHM)
print(f"Concept labels collected: {concept_result.num_samples} samples")
print(f"Concepts: {list(concept_result.labels.keys())}")
for name, desc in concept_result.concept_descriptions.items():
    print(f"  {name}: {desc}")

# Save concept labels
torch.save({
    "labels": concept_result.labels,
    "descriptions": concept_result.concept_descriptions,
    "algorithm": concept_result.algorithm,
    "num_samples": concept_result.num_samples,
}, RESULTS_DIR / "concept_labels.pt")
print(f"\nSaved to {RESULTS_DIR / 'concept_labels.pt'}")

### 2.4 Train BatchTopK SAE

In [ ]:
dict_size = HIDDEN_DIM * EXPANSION_FACTOR

sae = BatchTopKSAE(
    input_dim=HIDDEN_DIM,
    dict_size=dict_size,
    k=SAE_K,
).to(device)

print(f"BatchTopK SAE: input_dim={HIDDEN_DIM}, dict_size={dict_size}, k={SAE_K}")
print(f"SAE parameters: {sum(p.numel() for p in sae.parameters()):,}")

In [ ]:
trainer = SAETrainer(sae, lr=SAE_LR, resample_dead_every=25_000)
sae_loader = make_activation_dataloader(activations, batch_size=SAE_BATCH_SIZE)

sae_losses = []
step = 0
pbar = tqdm(total=SAE_STEPS, desc="Training SAE")

while step < SAE_STEPS:
    for (batch,) in sae_loader:
        if step >= SAE_STEPS:
            break
        batch = batch.to(device)
        output = trainer.train_step(batch)

        if step % 500 == 0:
            sae_losses.append({
                'step': step,
                'loss': output.loss.item(),
                'recon': output.reconstruction_loss.item(),
                'l0': output.l0.item(),
            })
            pbar.set_postfix(
                loss=f"{output.loss.item():.4f}",
                recon=f"{output.reconstruction_loss.item():.4f}",
                L0=f"{output.l0.item():.1f}",
            )

        step += 1
        pbar.update(1)

pbar.close()
print(f"\nFinal training stats: {trainer.get_training_stats()}")

In [ ]:
# Save SAE checkpoint
torch.save({
    "state_dict": sae.state_dict(),
    "config": sae.get_config(),
    "sae_type": SAE_TYPE,
}, RESULTS_DIR / "sae.pt")
print(f"Saved SAE to {RESULTS_DIR / 'sae.pt'}")

### 2.5 SAE training curves

In [ ]:
steps = [d['step'] for d in sae_losses]
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

ax1.plot(steps, [d['loss'] for d in sae_losses])
ax1.set_xlabel("Step")
ax1.set_ylabel("Total Loss")
ax1.set_title("SAE Total Loss")

ax2.plot(steps, [d['recon'] for d in sae_losses])
ax2.set_xlabel("Step")
ax2.set_ylabel("Reconstruction Loss")
ax2.set_title("SAE Reconstruction Loss")

ax3.plot(steps, [d['l0'] for d in sae_losses])
ax3.set_xlabel("Step")
ax3.set_ylabel("L0 (active features)")
ax3.set_title("SAE Sparsity (L0)")
ax3.axhline(y=SAE_K, color='r', linestyle='--', alpha=0.5, label=f"target k={SAE_K}")
ax3.legend()

plt.tight_layout()
plt.show()

---
## Phase 3: Feature Interpretation

Analyze SAE features: basic statistics, reconstruction quality, and — most importantly — correlations with ground-truth algorithmic concepts (is_visited, is_frontier, etc.).

### 3.1 Reconstruction quality & dead features

In [ ]:
sae.eval()
analyzer = FeatureAnalyzer(sae)

# Reconstruction quality
with torch.no_grad():
    sample = activations[:10_000].to(device)
    output = sae(sample)
    reconstructed = output.reconstructed

    recon_mse = output.reconstruction_loss.item()
    total_var = sample.var(dim=0).sum()
    residual_var = (sample - reconstructed).var(dim=0).sum()
    fve = (1 - residual_var / total_var).item()

print(f"Reconstruction MSE: {recon_mse:.6f}")
print(f"Fraction of variance explained: {fve:.4f}")

# Dead features
with torch.no_grad():
    dead_mask = sae.get_dead_features(sample)
    num_dead = dead_mask.sum().item()
print(f"Dead features: {num_dead}/{dict_size} ({100*num_dead/dict_size:.1f}%)")

# L0 sparsity
with torch.no_grad():
    features = sae.encode(sample)
    l0_per_sample = (features > 0).float().sum(dim=-1)
    print(f"L0 (avg active features): {l0_per_sample.mean():.1f} +/- {l0_per_sample.std():.1f}")
    print(f"L0 range: [{l0_per_sample.min():.0f}, {l0_per_sample.max():.0f}]")

### 3.2 Feature statistics

In [ ]:
stats = analyzer.compute_feature_stats(activations.to(device))
stats_sorted = sorted(stats, key=lambda s: s.activation_frequency, reverse=True)

# Feature frequency distribution
freqs = [s.activation_frequency for s in stats]
active_freqs = [f for f in freqs if f > 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(freqs, bins=50, edgecolor='black', alpha=0.7)
ax1.set_xlabel("Activation Frequency")
ax1.set_ylabel("Count")
ax1.set_title("Feature Activation Frequency Distribution")
ax1.axvline(x=0, color='r', linestyle='--', alpha=0.5)

if active_freqs:
    ax2.hist(active_freqs, bins=50, edgecolor='black', alpha=0.7)
    ax2.set_xlabel("Activation Frequency")
    ax2.set_ylabel("Count")
    ax2.set_title("Active Features Only")

plt.tight_layout()
plt.show()

# Top-20 most active features
print("Top-20 most active features:")
for s in stats_sorted[:20]:
    print(f"  Feature {s.feature_idx:4d}: freq={s.activation_frequency:.4f}, "
          f"mean_act={s.mean_activation:.4f}, max_act={s.max_activation:.4f}")

### 3.3 Concept correlations — the core result

For each SAE feature, compute Pearson correlation with ground-truth BFS concepts:
- **is_visited**: node has been reached
- **is_frontier**: node just entered the wavefront this step
- **is_source**: source node
- **is_unvisited**: node not yet reached

In [ ]:
# Load concept labels
concept_data = torch.load(RESULTS_DIR / "concept_labels.pt", map_location=device, weights_only=True)
concept_labels = concept_data["labels"]

# Align dimensions: truncate to the smaller of activations / labels
min_n = min(activations.shape[0], min(v.shape[0] for v in concept_labels.values()))
acts_aligned = activations[:min_n].to(device)
labels_aligned = {k: v[:min_n].to(device) for k, v in concept_labels.items()}

print(f"Aligned samples: {min_n:,}")
print(f"Concepts: {list(labels_aligned.keys())}")

# Compute correlations
result = analyzer.compute_concept_correlations(acts_aligned, labels_aligned)
print(f"\nMonosemantic features: {len(result.monosemantic_features)} / {dict_size}")
print(f"Dead features: {len(result.dead_features)} / {dict_size}")

In [ ]:
# Top correlated features per concept
for concept_name in result.concept_names:
    top = analyzer.find_features_for_concept(
        acts_aligned, labels_aligned[concept_name], top_k=5
    )
    print(f"\nConcept '{concept_name}' — top correlated features:")
    for feat_idx, corr in top:
        print(f"  Feature {feat_idx:4d}: correlation = {corr:+.4f}")

### 3.4 Concept correlation heatmap

In [ ]:
# Heatmap of concept correlations for the top-K most interesting features
# Select features with highest max absolute correlation across any concept
concept_matrix = result.concept_matrix.cpu().numpy()  # (dict_size, num_concepts)
max_corr_per_feat = np.abs(concept_matrix).max(axis=1)
top_k_feats = 30
top_feat_idxs = np.argsort(max_corr_per_feat)[-top_k_feats:][::-1]

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(concept_matrix[top_feat_idxs], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)

ax.set_yticks(range(len(top_feat_idxs)))
ax.set_yticklabels([f"Feature {i}" for i in top_feat_idxs])
ax.set_xticks(range(len(result.concept_names)))
ax.set_xticklabels(result.concept_names, rotation=45, ha='right')
ax.set_title(f"Top-{top_k_feats} Features x BFS Concepts — Pearson Correlation")

plt.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

### 3.5 Monosemanticity analysis

A feature is **monosemantic** if it correlates strongly with exactly one concept. This is the key test: do SAE features cleanly separate algorithmic concepts?

In [ ]:
# Categorize features
mono_feats = result.monosemantic_features
dead_feats = result.dead_features
poly_feats = [i for i in range(dict_size) 
              if i not in mono_feats and i not in dead_feats 
              and max_corr_per_feat[i] > 0.1]
other_feats = [i for i in range(dict_size) 
               if i not in mono_feats and i not in dead_feats and i not in poly_feats]

print(f"Feature breakdown ({dict_size} total):")
print(f"  Monosemantic (|corr| > 0.5 with exactly 1 concept): {len(mono_feats)}")
print(f"  Polysemantic (|corr| > 0.1 with multiple concepts):  {len(poly_feats)}")
print(f"  Weak/uncorrelated:                                    {len(other_feats)}")
print(f"  Dead (never activate):                                {len(dead_feats)}")

# Detailed look at monosemantic features
if mono_feats:
    print(f"\nMonosemantic features detail:")
    for feat_idx in mono_feats[:15]:
        corrs = {name: concept_matrix[feat_idx, j] 
                 for j, name in enumerate(result.concept_names)}
        best_concept = max(corrs, key=lambda k: abs(corrs[k]))
        print(f"  Feature {feat_idx:4d} -> {best_concept} (r={corrs[best_concept]:+.3f})")

### 3.6 Max correlation distribution

Distribution of the maximum absolute correlation each feature achieves with any concept. High values = features that cleanly track algorithmic state.

In [ ]:
# Filter out dead features for the distribution
active_max_corrs = max_corr_per_feat[[i for i in range(dict_size) if i not in dead_feats]]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(active_max_corrs, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=0.5, color='r', linestyle='--', alpha=0.7, label="Monosemantic threshold (0.5)")
ax.axvline(x=0.3, color='orange', linestyle='--', alpha=0.7, label="Weak threshold (0.3)")
ax.set_xlabel("Max |correlation| with any concept")
ax.set_ylabel("Number of features")
ax.set_title("Feature-Concept Correlation Strength (active features)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary & Next Steps

**Key metrics to check:**
- NAR validation accuracy > 0.9 (BFS should be near-perfect)
- SAE fraction of variance explained > 0.9
- Dead features < 30%
- Monosemantic features > 0 (ideally 10+)

**Next experiments to run:**
- Repeat for Dijkstra, DFS, MST (change `ALGORITHM` at the top)
- Try different SAE expansion factors (1x, 4x, 16x) and k values
- Train Standard SAE baseline for comparison
- Multi-algorithm training (shared processor across BFS + Dijkstra + DFS)